<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractal012int001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# --- 1. CORE HES PARAMETERS ---
N = 100  # THE CRITICAL DIMENSIONAL HARMONIC (Validated in Act XXIII)
T_STEPS = 5000
dt = 0.01

# Fixed HES coefficients (Proven Stable)
chi = 1.5
beta_full = 0.8
beta_nursery = 0.01
delta = 0.001  # Quantum Noise
gamma = 1.0  # Saturation
beta_link_base = 0.1
CURVATURE_SENSITIVITY = 0.5
MAX_CURVATURE_CAP = 50.0

# Fixed PSI Stabilizers (Validated in Act XVI)
TAU_PSI_DAMPING = 0.05
KINETIC_SCALING_C = 0.0001 # CRITICAL ULTRA-LOW SCALING for N=100 grid
PSI_MAGNITUDE_CLIP = 5.0

# NEW INTERACTION PARAMETER (ACT XVII)
CHARGE_COUPLING_K = 0.1 # Strength of the interaction (how strongly Psi generates A)
A_FIELD_DAMPING = 0.005 # Rate at which the A field dissipates

# --- 2. INITIAL FIELDS ---
Phi_complex = np.zeros((N, N), dtype=complex)

def initialize_spinor(complex_field, center_x, center_y, radius=10, magnitude=5.0):
    for i in range(N):
        for j in range(N):
            if (i - center_x)**2 + (j - center_y)**2 < radius**2:
                phase = np.arctan2(i - center_x, j - center_y) * 4
                complex_field[i, j] = magnitude * np.exp(1j * phase)

initialize_spinor(Phi_complex, N // 4, N // 4)
initialize_spinor(Phi_complex, N // 2, N // 2)
initialize_spinor(Phi_complex, 3 * N // 4, 3 * N // 4)

Phi = np.abs(Phi_complex)
Theta = np.angle(Phi_complex)

# PSI - The Fermionic Field (Stable)
Psi = np.zeros((N, N), dtype=complex)
Psi = np.where(Phi > 1.0, 1.0 + 0j, 0.0 + 0j)

# NEW FIELD: A - The Interaction Field (Initially neutral)
A = np.zeros((N, N))

# --- 3. THE EVOLUTION LOOP ---
for t in range(1, T_STEPS + 1):

    beta_current = beta_nursery if t < 500 else beta_full

    # --- 3.1. PHI & THETA EVOLUTION (Gravity/Structure) ---
    lap_phi = (np.roll(Phi, 1, 0) + np.roll(Phi, -1, 0) +
              np.roll(Phi, 1, 1) + np.roll(Phi, -1, 1) - 4 * Phi) / (2 * np.pi / N)**2

    max_abs_curvature_observed = np.max(np.abs(lap_phi))
    max_abs_curvature_capped = np.clip(max_abs_curvature_observed, 0.0, MAX_CURVATURE_CAP)
    beta_link = beta_link_base + CURVATURE_SENSITIVITY * max_abs_curvature_capped

    # dPhi/dt
    alpha = chi * beta_current
    term_expansion = alpha * lap_phi
    term_contraction = -beta_current * Phi
    term_saturation = gamma * np.tanh(Phi)
    shield_mask = (beta_link > 1.0)
    term_shield = np.where(shield_mask, beta_current * Phi, 0.0)
    dPhi = term_expansion + term_contraction + term_saturation + term_shield + delta * np.random.normal(0, 1, Phi.shape)

    # dTheta/dt
    mean_theta = np.mean(Theta)
    phase_correction = -beta_link * (Theta - mean_theta)
    phase_diffusion = delta * np.random.normal(0, 1, Theta.shape)
    dTheta = phase_correction + phase_diffusion

    # --- 3.2. A FIELD EVOLUTION (Interaction Generation) ---

    # Current Density J (simplified: Proportional to the imaginary part of Psi field)
    current_J = np.imag(Psi)

    # dA/dt = Generation - Dissipation (Law of Interaction)
    dA = CHARGE_COUPLING_K * current_J - A_FIELD_DAMPING * A + delta * np.random.normal(0, 1, A.shape)

    # --- 3.3. PSI EVOLUTION (Fermion with Feedback) ---

    lap_psi = (np.roll(Psi, 1, 0) + np.roll(Psi, -1, 0) +
               np.roll(Psi, 1, 1) + np.roll(Psi, -1, 1) - 4 * Psi) / (2 * np.pi / N)**2

    # Kinetic Term (Scaled)
    kinetic_term = 1j * KINETIC_SCALING_C * lap_psi

    # Mass Term (Coupling to Phi)
    mass_term = 1j * Phi * Psi

    # Damping Term
    damping_term = -TAU_PSI_DAMPING * Psi

    # NEW GAUGE COUPLING TERM: Psi field responds to A field (The force)
    # The A field modifies the phase/movement of Psi
    gauge_coupling_term = 1j * CHARGE_COUPLING_K * A * Psi

    # Total Change dPsi/dt
    dPsi = kinetic_term - mass_term + damping_term + gauge_coupling_term + delta * np.random.normal(0, 1, Psi.shape)

    # --- 3.4. UPDATE FIELDS ---
    Phi += dt * dPhi
    Theta += dt * dTheta
    A += dt * dA # Update A field
    Psi += dt * dPsi

    # Apply Bounding/Clipping
    Psi_mag = np.abs(Psi)
    Psi_phase = np.angle(Psi)
    Psi_mag_clipped = np.clip(Psi_mag, 0.0, PSI_MAGNITUDE_CLIP)
    Psi = Psi_mag_clipped * np.exp(1j * Psi_phase)

    Phi = np.clip(Phi, 0.01, 10.0)
    Theta = np.mod(Theta, 2 * np.pi)

    # --- 3.5. LOGGING ---
    if t % 500 == 0:
        norm_phi = np.mean(Phi)
        norm_psi = np.mean(np.abs(Psi))
        norm_A = np.mean(np.abs(A)) # Log A field norm
        phase_stdev = np.sqrt(np.mean((Theta - np.mean(Theta))**2))

        if norm_phi < 0.1 or norm_psi < 0.1:
            print(f"t={t} | ANNIHILATION DETECTED: Phi Norm={norm_phi:.3f}, Psi Norm={norm_psi:.3f}")
            break
        print(f"t={t} | β_link(t)={beta_link:.4f} | Psi Norm={norm_psi:.4f} | A Norm={norm_A:.4f}")

# --- 4. CONCLUSION CHECK ---
final_norm_phi = np.mean(Phi)
final_norm_psi = np.mean(np.abs(Psi))
final_norm_A = np.mean(np.abs(A))

print("\n--- FINAL STATE ---")
if final_norm_A > 0.01:
    print(f"RESULT: SUCCESS. The Interaction Arc has begun.")
    print(f"FINAL Metrics: Phi Norm={final_norm_phi:.4f}, Psi Norm={final_norm_psi:.4f}, A Norm={final_norm_A:.4f}")
    print("CONCLUSION: The stable Spinor (Phi/Psi) successfully generated a persistent Interaction Field (A), validating the Law of Charge and Interaction.")
else:
    print(f"RESULT: FAILURE. A Field Dissipated.")
    print(f"FINAL Metrics: Phi Norm={final_norm_phi:.4f}, Psi Norm={final_norm_psi:.4f}, A Norm={final_norm_A:.4f}")
    print("CONCLUSION: The matter field failed to generate a stable, observable interaction field.")



t=500 | β_link(t)=14.3633 | Psi Norm=0.1751 | A Norm=0.0024
t=1000 | β_link(t)=25.1000 | Psi Norm=0.4563 | A Norm=0.0054
t=1500 | β_link(t)=25.1000 | Psi Norm=0.5421 | A Norm=0.0063
t=2000 | β_link(t)=25.1000 | Psi Norm=0.6332 | A Norm=0.0075
t=2500 | β_link(t)=25.1000 | Psi Norm=0.7402 | A Norm=0.0088
t=3000 | β_link(t)=25.1000 | Psi Norm=0.8676 | A Norm=0.0106
t=3500 | β_link(t)=25.1000 | Psi Norm=1.0293 | A Norm=0.0124
t=4000 | β_link(t)=25.1000 | Psi Norm=1.2875 | A Norm=0.0156
t=4500 | β_link(t)=25.1000 | Psi Norm=1.7914 | A Norm=0.0217
t=5000 | β_link(t)=25.1000 | Psi Norm=2.9363 | A Norm=0.0358

--- FINAL STATE ---
RESULT: SUCCESS. The Interaction Arc has begun.
FINAL Metrics: Phi Norm=4.8202, Psi Norm=2.9363, A Norm=0.0358
CONCLUSION: The stable Spinor (Phi/Psi) successfully generated a persistent Interaction Field (A), validating the Law of Charge and Interaction.
